In [1]:

from datetime import datetime
import random
import pickle
import gzip
import os, sys
import wget
import numpy as np
import matplotlib
matplotlib.use('Agg')

import warnings
warnings.filterwarnings('ignore')

import copy
import time
from collections import OrderedDict
import pandas as pd

import torch

from network.SmilesImage2Smiles_Network_vae import *
from data_utils import *
from data_utils_smiles import *

%load_ext autoreload
%autoreload 2

### **Parameters**

In [2]:
path_init = '/home/rkmvu/Dataset/selfies/zinc/'
gpu_device_id = 0# GPU number for multiple GPUs (pytorch takes default 0 or which is availabe next)
device = torch.device("cuda:"+str(gpu_device_id) if torch.cuda.is_available() else "cpu")
# print(torch.cuda.is_available())
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('='*70)
print('Model will use: {}' .format(device))
print('='*70)


# dataset parameters
# num_total_samples = 3963360#3970176
num_train_samples = 12941195#12941195#10500000#100000#10#
num_train_test_samples = 100000#100000#'all'#10#
train_ratio = 0.5
val_ratio = 0.2
test_ratio = 1 - train_ratio - val_ratio

selected_data_frac = 0.7
ticks = [-1, 0, 1, 2, 3, 4, 5]
# dataset_name_prefix = 'zinc6m_all4'#'zinc6m_all4_mona'
smiles_char_filename = 'tokens_smiles_train_0_5_val_0_2_test_0_3_63.json'
smiles_max_length_filename = 'max_len_smiles_train_0_5_val_0_2_test_0_3_241.txt'
dataset_name = 'canon_smiles_selfies_with_descriptors_train_0_5_val_0_2_test_0_3.parquet'
caco2_all_data_path_filename = 'all_data_features_modified_wrt_independent_set.csv'
caco2_independent_data_path_filename = 'independent_test_set.csv'
padding = 'right'

train_data_name = ''.join(['train_', str(train_ratio), '_']).replace('.', '_')
test_data_name = ''.join(['test_', str(test_ratio), '_']).replace('.', '_')
val_data_name = ''.join(['val_', str(val_ratio), '_']).replace('.', '_')

smiles_char_filepath = ''.join([path_init, '/', smiles_char_filename])
smiles_max_length_filepath = ''.join([path_init, '/', smiles_max_length_filename])
#+++++++++++++++++++++++++++++++++++++++++++++++++++

# read smiles,  vocabulary and smiles max-length
smiles = PARSE_SMILES([])

padding = padding
# load smiles char vocabulary
smiles.smiles_char, smiles.smiles_char_map = smiles.load_smiles_char(smiles_char_filepath)
# load smiles max-length
smiles.max_smiles_length = smiles.load_smiles_max_length(smiles_max_length_filepath)
#++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Model will use: cuda:0


### **Creating and Loading model**

In [3]:
## VAE_GRU parameters
dims_input_data = (smiles.max_smiles_length, len(smiles.smiles_char_map['domain'].keys())+1) 
dims_output_data = (smiles.max_smiles_length, len(smiles.smiles_char_map['domain'].keys())+1)
num_kernels = [9, 9, 10]#[[11, 13, 15]#11, 13, 15]#[15, 17, 19]#[11, 13, 15]#[9, 9, 10]
size_kernels = [9, 9, 11]#[[13, 11, 9]#9, 9, 11]#[15, 13, 11]#[9, 9, 11]#[9, 9, 11]
num_fc_layer_encoder = 0#2#
dropout_prob = 0.0
dims_latent = 300#200#100#50#25#
act_func = 'relu'
scale_latent_space = 1e-2
num_fc_layer_decoder = 0
gru_hidden_size = 500
num_gru = 4#5#3#
layer_type = '1dcnn_gru'
weight_init = 'xvr_unifrm'
loss_type = 'bce_kld'
solver_type = 'adam'
num_epoch = 500
batch_size = 128
learning_rate = 1e-4#1e-3#
save_result_ateach_epoch = 50
result_savepath = ''.join(['cache/smi2smi_vae/', path_init.split('/')[-2], '_', dataset_name, '_', str(num_train_samples), '/']).replace('.','_')
#+++++++++++++++++++++++++++++++++++++++++++++++++++

smiles_vae_model = SmIm2Sm_Network(dims_input_data, dims_output_data, 
                                   smiles_char_map=smiles.smiles_char_map, 
                                   max_string_len=smiles.max_smiles_length, 
                                   padding=padding, 
                                   num_kernels=num_kernels, 
                                   size_kernels=size_kernels, 
                                   num_fc_layer_encoder=num_fc_layer_encoder, 
                                   dropout_prob=dropout_prob, 
                                   dims_latent=dims_latent, 
                                   act_func=act_func, 
                                   scale_latent_space=scale_latent_space, 
                                   num_fc_layer_decoder=num_fc_layer_decoder, 
                                   gru_hidden_size=gru_hidden_size, 
                                   num_gru=num_gru, 
                                   layer_type=layer_type, 
                                   device=device, 
                                   weight_init=weight_init, 
                                   loss_type=loss_type, 
                                   solver_type=solver_type, 
                                   num_epoch=num_epoch, 
                                   batch_size=batch_size, 
                                   learning_rate=learning_rate, 
                                   save_result_ateach_epoch=save_result_ateach_epoch, 
                                   result_savepath=result_savepath)
#+++++++++++++++++++++++++++++++++++++++++++++++++++


# test on best trained network
smiles_vae_model.load_test_network()# load pre-trained best network


----------------------------------------------------------------------
Network summary
----------------------------------------------------------------------
Weights "Conv1d(241, 9, kernel_size=(9,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Conv1d(9, 9, kernel_size=(9,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Conv1d(9, 10, kernel_size=(11,), stride=(1,))" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=380, out_features=300, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=300, out_features=300, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=300, out_features=300, bias=True)" initialized by "xavior_uniform" scheme
Weights "Linear(in_features=300, out_features=300, bias=True)" initialized by "xavior_uniform" scheme
Weights "GRU(300, 500, num_layers=4, batch_first=True)" initialized by "orthogonal" scheme
Weights "Linear(in_features=500, out_features=64, bias=True)

## **Test the model for representation**

In [4]:
def encode(smiles_or_selfies):
    z, _, _ =  smiles_vae_model.smiles2latentspace_representation(smiles=smiles_or_selfies, 
                                                            verbose=1)
    return z

def decode(x_latent):
    x = smiles_vae_model.latent_space2smiles(x_latent=x_latent)
    return x

def name2smiles(name):
    row = df_temp[df_temp['name'] == name].iloc[0]
    return row['smiles']

def smiles2name(smiles):
    smiles = canonical_fn(smiles)
    row = df_temp[df_temp['smiles'] == smiles].iloc[0]
    return row['name']

def smi2nneigh(smiles, indices, n_neigh=10):
    smiles = canonical_fn(smiles)
    idx = smiles_main.index(smiles)
    neigh_idx = indices[idx][1:n_neigh+1]
    nn_smiles = [smiles_main[i] for i in neigh_idx]
    nn_names = [smiles2name(x) for x in nn_smiles]
    return {'smiles':nn_smiles, 'name':nn_names}


In [5]:
import pandas as pd

smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)

df = pd.read_csv('/home/rkmvu/Dataset/selfies/zinc/properties_dtratio_0.001_test_0_3_canon_smiles_selfies_with_descriptors_train_0_5_val_0_2_test_0_3.csv')
mask = df['SMILES'].apply(smiles_clean_fn)
smiles_main = df[mask]['SMILES'].tolist()
df

,SMILES,MolWt,TPSA,EState_VSA1,NHOHCount,MolLogP,fr_COO,nAcid,ATSC1c,ATSC1se,...,fr_ether,fr_halogen,fr_ketone,fr_nitro,fr_nitro_arom,fr_nitroso,fr_phenol,fr_sulfone,AUTOCORR2D_156,nBase
0,COc1ccc(CN2CCC3CN(C(=O)C(C)C(C)(C)C)C3C2)nn1,346.475,58.56,0.000000,0,2.20010,0,0,-0.330215,0.267548,...,1,0,0,0,0,0,0,0,1.405,1
1,C#CCOC(C)C(=O)N1CC2(CCCN2C(=O)c2ccc(CO)o2)C1,346.383,83.22,6.103966,1,0.62720,0,0,-0.612666,-0.332817,...,1,0,0,0,0,0,0,0,1.405,0
2,CNC(=O)c1ccc(C)c(NC(=O)NC(C)c2ccccc2OCc2ccccc2)c1,417.509,79.46,0.000000,3,4.81632,0,0,-0.750331,-0.201391,...,1,0,0,0,0,0,0,0,1.476,0
3,COC1(C)CCN(C(=O)c2ccccc2SC(F)(F)F)CC1,333.375,29.54,5.508331,0,3.93960,0,0,-0.420343,-0.167045,...,1,3,0,0,0,0,0,0,1.182,0
4,CS(=O)(=O)CCC1NC(=O)N(CCOc2cccc(Cl)c2)C1=O,360.819,92.78,27.817388,1,1.07390,0,0,-0.617955,0.009875,...,1,1,0,0,0,0,0,1,1.245,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7759,CC(C)NCCC(=O)N1CCCCC1c1cc(NC(=O)c2cn3cc(Cl)ccc...,457.966,107.42,0.000000,3,3.40480,0,0,-0.588136,-0.127373,...,0,1,0,0,0,0,0,0,0.835,1
7760,N#Cc1ccc(N2CCCC2C(=O)OCC(=O)Nc2ccc([N+](=O)[O-...,462.384,125.57,52.563041,1,3.63598,0,0,-0.679651,0.251615,...,1,3,0,1,1,0,0,0,1.187,0
7761,CCC(N)CNC(=O)NC(C)CCc1ccc2c(c1)OCO2,307.394,85.61,0.000000,4,1.77290,0,0,-0.794791,-0.296837,...,2,0,0,0,0,0,0,0,1.252,1
7762,COc1cc(C(=O)N2CCC3CCN(Cc4cnsn4)C3C2)sn1,365.484,71.45,0.000000,0,1.73980,0,0,-0.352193,0.126301,...,1,0,0,0,0,0,0,0,0.880,1


In [9]:
z = encode(smiles_or_selfies=smiles_main)
smiles_recon = decode(x_latent=z)

exact = [x==y for x, y in zip(smiles_main, smiles_recon)]
smis, checks = smiles.check_smiles_validity(smiles=smiles_recon)

print('='*80)
print('This is a subset of the full test dataset')
print(f'Number of SMILES: {len(smiles_main)}')
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


 17%|█▋        | 9/54 [00:00<00:00, 80.52it/s]

<===smiles to latent space representation===>


 33%|███▎      | 18/54 [00:00<00:00, 83.66it/s]

100%|██████████| 54/54 [00:00<00:00, 65.23it/s]


<===smiles reconstruction===>


100%|██████████| 54/54 [00:07<00:00,  7.32it/s]
6813it [00:01, 4481.85it/s]

This is a subset of the full test dataset
Number of SMILES: 6813
Exact reconstruction: 0.9621312197269926
Valid reconstruction: 0.9831205049170703


## **Test the model for representation**

In [10]:
import selfies as sf
import pandas as pd

smiles2selfie_fn = lambda x: sf.encoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
len_filter_fn = lambda x: sf.len_selfies(x)<=128
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)


#### **Preprocess drugs**

In [11]:
df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = smiles.check_smiles_validity(df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
smiles_main = df_temp['smiles'].tolist()
print(f'Number of smiles: {len(smiles_main)}')
df_temp

998it [00:00, 5014.42it/s]

1381it [00:00, 4879.54it/s]


Number of smiles: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


### **Finding Nearest Neighbours**

In [12]:
from sklearn.neighbors import NearestNeighbors

z = encode(smiles_or_selfies=smiles_main)
print(z.shape)
n_neigh = NearestNeighbors(n_neighbors=60, metric='euclidean')
n_neigh.fit(z)

100%|██████████| 9/9 [00:00<00:00, 98.71it/s]

<===smiles to latent space representation===>
(1089, 300)


NearestNeighbors(metric='euclidean', n_neighbors=60)

In [13]:
distances, indices = n_neigh.kneighbors(z)
distances
indices

array([[   0,  235,  718, ...,  557,  219,  154],
       [   1,  615,  440, ...,   22,  656,   24],
       [   2,   12,  841, ...,  371,  150,   43],
       ...,
       [1086,   69,   67, ...,  671,  672,  628],
       [1087,  788,   98, ...,  604,  648,  631],
       [1088,  787,  869, ...,  793, 1050, 1058]])

In [ ]:
df = pd.DataFrame(z, columns=[f'dim_{i}' for i in range(z.shape[1])])
df.insert(0, 'smiles', smiles_main)
df.insert(0, 'selfies', 'None')
df.insert(0, 'name', 'None')
df['name'] = df['smiles'].apply(smiles2name)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

df2 = pd.read_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_properties.csv')
df3 = pd.merge(df, df2, on='smiles')
# df3.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_props_with_embeds_evsmi.csv', index=False)
df3.head(2)

,name,selfies,smiles,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,VSA_EState3,NHOHCount,NumHDonors,NumHAcceptor,NumRotatableBonds,MolLogP,ATSC1pe,ATSC1are,AATSC1dv,AATSC1are
0,Abacavir,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,-0.013574,-0.000104,0.068762,-0.004215,-0.006102,0.042947,-0.017929,...,12.615719,4,3,7,4,1.0923,-0.478300,-0.657070,1.039823,-0.015645
1,Abiraterone,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,-0.052832,0.014818,-0.013556,0.022417,0.018578,0.002396,-0.003495,...,0.000000,0,0,3,2,5.9694,0.296168,0.241524,1.157286,0.003659


In [16]:
name2smiles('Abacavir')

'Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1'

In [17]:
smiles2name('Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1')

'Abacavir'

In [18]:
smi2nneigh(name2smiles('Loxapine'), indices=indices, n_neigh=3)

{'smiles': ['CN1CCN(CCCN2c3ccccc3Sc3ccc(Cl)cc32)CC1',
  'CN1CCC(=C2c3ccccc3CCn3c(C=O)cnc32)CC1',
  'CN1CCC(=C2c3ccccc3CC(=O)c3sccc32)CC1'],
 'name': ['Prochlorperazine', 'Alcaftadine', 'Ketotifen']}

### **Reconstruction accuracy**

In [19]:
z = encode(smiles_or_selfies=smiles_main)
smiles_recon = decode(x_latent=z)
print(z.shape)

exact = [x==y for x, y in zip(smiles_main, smiles_recon)]
smis, checks = smiles.check_smiles_validity(smiles=smiles_recon)

print('='*80)
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


100%|██████████| 9/9 [00:00<00:00, 93.47it/s]


<===smiles to latent space representation===>
<===smiles reconstruction===>


 11%|█         | 1/9 [00:00<00:00,  8.25it/s]

100%|██████████| 9/9 [00:01<00:00,  8.36it/s]


(1089, 300)


1089it [00:00, 4532.62it/s]

Exact reconstruction: 0.4738292011019284
Valid reconstruction: 0.5977961432506887


## **Nearest Neighbour Analysis**

In [20]:
import numpy as np

top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=50

all_out = {'drug_evsmi':[], 'nn_idx_evsmi':[], 'smiles_evsmi':[], 'names_evsmi':[]}
for drug in top_drugs:
    _smiles = name2smiles(drug)
    out = smi2nneigh(_smiles, indices=indices, n_neigh=n_neigh)
    drug_name = [drug]*n_neigh
    idx = np.arange(1, n_neigh+1)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_evsmi'].extend(out['drug'])
    all_out['nn_idx_evsmi'].extend(out['nn_idx'])
    all_out['smiles_evsmi'].extend(out['smiles'])
    all_out['names_evsmi'].extend(out['name'])

In [25]:
all_out_df = pd.DataFrame(all_out)
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_evsmi.csv', index=False)
all_out_df


,drug_evsmi,nn_idx_evsmi,smiles_evsmi,names_evsmi
0,Abacavir,1,Nc1nc(Cl)nc2c1ncn2C1CC(O)C(CO)O1,Cladribine
1,Abacavir,2,COc1nc(N)nc2c1ncn2C1OC(CO)C(O)C1O,Nelarabine
2,Abacavir,3,N#CC1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,Vildagliptin
3,Abacavir,4,Cc1cc(-c2ccccc2)nnc1NCCN1CCOCC1,Minaprine
4,Abacavir,5,Clc1ccc(COC(Cn2ccnc2)c2ccc(Cl)cc2Cl)cc1,Econazole
...,...,...,...,...
54445,Zuclopenthixol,46,Clc1ccc(COC(Cn2ccnc2)c2ccc(Cl)cc2Cl)c(Cl)c1,Miconazole
54446,Zuclopenthixol,47,Oc1ccc(CCCCNCC(O)c2ccc(O)c(O)c2)cc1,Arbutamine
54447,Zuclopenthixol,48,COc1cc(NC(C)CCCN)c2ncccc2c1,Primaquine
54448,Zuclopenthixol,49,CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1,Loratadine


In [22]:
all_out_df['drug_evsmi'].nunique()

1089